# EDA Methodology: A Systematic Approach

**Module:** 1 — Foundations & Data Engineering  
**Week:** 2, Day 2  

## Learning Objectives
- Follow a repeatable 5-step EDA framework
- Analyze distributions, correlations, and outliers
- Visualize effectively with Matplotlib and Seaborn
- Spot data quality issues systematically
- Translate findings into business insights

## Resources
- [Kaggle Learn: Data Visualization](https://www.kaggle.com/learn/data-visualization)
- [Seaborn Tutorial](https://seaborn.pydata.org/tutorial.html)
- [Matplotlib Cheat Sheet](https://matplotlib.org/cheatsheets/)

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from shared.viz_utils import setup_style
from shared.data_utils import describe_df
setup_style()

# We'll reuse our sales data generator
sys.path.insert(0, '../projects/p1-sales-pipeline')
from src.generate_data import generate_sales_data

df = generate_sales_data(n_orders=2000)
print(f"Loaded {len(df)} rows")

---
## The 5-Step EDA Framework

1. **Shape & Schema** — What does the data look like?
2. **Quality Audit** — Missing values, duplicates, inconsistencies
3. **Univariate Analysis** — Each variable on its own
4. **Bivariate/Multivariate** — Relationships between variables
5. **Key Findings** — Summarize insights for stakeholders

---
## Step 1: Shape & Schema

In [ ]:
# Quick overview
describe_df(df)

In [ ]:
# Detailed info
df.info()

In [ ]:
# Numeric summary
df.describe().round(2)

In [ ]:
# Categorical summary
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head())

---
## Step 2: Quality Audit

In [ ]:
# Missing values summary
missing = pd.DataFrame({
    'count': df.isnull().sum(),
    'pct': (df.isnull().mean() * 100).round(1)
}).query('count > 0').sort_values('pct', ascending=False)

if len(missing) > 0:
    print("Missing Values:")
    print(missing)
else:
    print("No missing values!")

In [ ]:
# Duplicates
dupe_count = df.duplicated().sum()
print(f"Exact duplicate rows: {dupe_count}")

if dupe_count > 0:
    print("\nSample duplicates:")
    dupes = df[df.duplicated(keep=False)].sort_values(list(df.columns))
    print(dupes.head(10))

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(series: pd.Series, multiplier: float = 1.5) -> pd.Series:
    """Return a boolean mask of outliers using the IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    return (series < lower) | (series > upper)

numeric_cols = df.select_dtypes(include=np.number).columns
print("Outlier counts (IQR method):")
for col in numeric_cols:
    n_outliers = detect_outliers_iqr(df[col].dropna()).sum()
    if n_outliers > 0:
        print(f"  {col}: {n_outliers} outliers ({n_outliers/len(df)*100:.1f}%)")

---
## Step 3: Univariate Analysis

Understand each variable individually.

In [ ]:
# Numeric distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols[:6]):
    ax = axes[i]
    df[col].dropna().hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col)
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'mean={df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='green', linestyle='--', label=f'median={df[col].median():.1f}')
    ax.legend(fontsize=8)

plt.suptitle('Numeric Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots — spot outliers visually
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['revenue', 'profit', 'quantity']):
    sns.boxplot(data=df, y=col, ax=axes[i])
    axes[i].set_title(f'{col} Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Categorical variable counts
cat_cols = ['category', 'product', 'region']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    counts.plot.bar(ax=axes[i], edgecolor='black')
    axes[i].set_title(f'{col} Counts')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## Step 4: Bivariate & Multivariate Analysis

Explore relationships between variables.

In [ ]:
# Correlation matrix for numeric columns
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue by category — box plot shows distribution, not just average
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='category', y='revenue', ax=ax)
ax.set_title('Revenue Distribution by Category')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue by category AND region — grouped comparison
fig, ax = plt.subplots(figsize=(12, 5))
(
    df.dropna(subset=['region'])
    .groupby(['category', 'region'])['revenue']
    .sum()
    .unstack()
    .plot.bar(ax=ax, edgecolor='black')
)
ax.set_title('Total Revenue by Category & Region')
ax.set_ylabel('Revenue ($)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Region')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot — unit_price vs quantity, colored by category
fig, ax = plt.subplots(figsize=(10, 6))
for cat in df['category'].unique():
    subset = df[df['category'] == cat]
    ax.scatter(subset['unit_price'], subset['quantity'], alpha=0.5, label=cat, s=30)

ax.set_xlabel('Unit Price ($)')
ax.set_ylabel('Quantity')
ax.set_title('Price vs Quantity by Category')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Time series — revenue trend over time
df['date'] = pd.to_datetime(df['date'])
monthly = df.groupby(df['date'].dt.to_period('M')).agg(
    revenue=('revenue', 'sum'),
    profit=('profit', 'sum'),
    orders=('order_id', 'count')
).reset_index()
monthly['date'] = monthly['date'].dt.to_timestamp()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(monthly['date'], monthly['revenue'], 'b-o', label='Revenue')
ax1.plot(monthly['date'], monthly['profit'], 'g-s', label='Profit')
ax1.set_title('Monthly Revenue & Profit Trend')
ax1.set_ylabel('Amount ($)')
ax1.legend()

ax2.bar(monthly['date'], monthly['orders'], width=20, edgecolor='black', alpha=0.7)
ax2.set_title('Monthly Order Count')
ax2.set_ylabel('Orders')
ax2.set_xlabel('Month')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap — average revenue by product × region
pivot = df.dropna(subset=['region']).pivot_table(
    values='revenue', index='product', columns='region', aggfunc='mean'
).round(0)

fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Avg Revenue: Product × Region')
plt.tight_layout()
plt.show()

In [ ]:
# Discount impact analysis
df['has_discount'] = df['discount'].fillna(0) > 0

discount_impact = df.groupby('has_discount').agg(
    avg_revenue=('revenue', 'mean'),
    avg_quantity=('quantity', 'mean'),
    avg_profit=('profit', 'mean'),
    count=('order_id', 'count')
).round(2)

print("Impact of Discounts:")
discount_impact

---
## Step 5: Key Findings

Summarize your findings in plain business language. Practice writing like you're presenting to a non-technical stakeholder.

### Template:

**Finding:** [What you observed]  
**Evidence:** [Numbers/chart that support it]  
**Implication:** [What this means for the business]  
**Recommendation:** [What action to take]

In [ ]:
# TODO: Write 3-5 key findings from your EDA
#
# Example:
# Finding 1:
#   Observation: Electronics category drives 40% of total revenue
#   Evidence: Total electronics revenue = $X vs overall $Y
#   Implication: Heavy dependency on one category
#   Recommendation: Diversify product mix or double down on electronics marketing

---
## Visualization Cheat Sheet

| Question | Chart Type | Seaborn/Matplotlib |
|----------|-----------|--------------------|
| Distribution of one numeric var | Histogram, KDE | `sns.histplot()`, `sns.kdeplot()` |
| Spread + outliers | Box plot, Violin | `sns.boxplot()`, `sns.violinplot()` |
| Counts of categories | Bar chart | `sns.countplot()`, `.value_counts().plot.bar()` |
| Two numeric vars | Scatter plot | `sns.scatterplot()`, `plt.scatter()` |
| Numeric by category | Box/Violin | `sns.boxplot(x='cat', y='num')` |
| Trend over time | Line chart | `plt.plot()`, `sns.lineplot()` |
| Correlation | Heatmap | `sns.heatmap(df.corr())` |
| Two categories vs numeric | Grouped bar, Heatmap | `.pivot_table().plot.bar()` |
| Part of whole | Pie (sparingly), stacked bar | `plt.pie()`, `.plot.bar(stacked=True)` |

### Style Tips
- Always add titles and axis labels
- Use `plt.tight_layout()` to prevent overlap
- Limit colors to 5-7 in one chart
- Horizontal bar charts are easier to read for many categories
- Annotate key data points when presenting to stakeholders

---
## Practice Exercises

### Exercise 1: Discount Deep Dive
Analyze the discount column:
- What % of orders have a discount?
- What's the most common discount level?
- How do discounted vs non-discounted orders compare on profit margin?
- Create a visualization showing discount impact on profitability by category.

In [ ]:
# Exercise 1: Your code here


### Exercise 2: Customer Behavior
Group by `customer_id` and analyze:
- Distribution of orders per customer
- Top 10 vs bottom 10 customers by revenue
- Is there a relationship between order frequency and average order value?
- Create a scatter plot of order_count vs avg_order_value.

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Presentation Chart
Create ONE polished chart that tells a compelling story from this data. This should be presentation-ready:
- Clear title
- Labeled axes
- Annotations on key data points
- A 2-sentence caption explaining the insight

In [ ]:
# Exercise 3: Your polished chart here


---
## Key Takeaways

1. **Follow a framework** — don't just make random charts. Shape → Quality → Univariate → Bivariate → Findings.
2. **Always check data quality first** — missing values, duplicates, and outliers affect every analysis downstream.
3. **Distribution matters** — mean can be misleading. Always look at median, spread, and shape.
4. **Visualize, don't just compute** — a correlation number is good; a scatter plot tells the whole story.
5. **Write findings in business language** — Finding + Evidence + Implication + Recommendation.